# 03 — Inventory & production optimization

Two practical questions for the kitchen:

1. **What should we bake tomorrow?** — Forecast demand per product from the last 30 days.
2. **Which raw materials are at risk?** — Compare on-hand stock to projected consumption.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from db import load_sale_items, load_stock, read_sql

items = load_sale_items()
stock = load_stock()

recipes = read_sql('''
    SELECT ri.product_id, ri.raw_material_id, ri.quantity,
           rm.name AS material_name, rm.unit AS material_unit, rm.unit_cost
    FROM catalog_recipeitem ri
    JOIN catalog_rawmaterial rm ON rm.id = ri.raw_material_id
''')
len(items), len(recipes), len(stock)

## 1. Demand forecast per product (last 30 days × weekday avg)

In [ ]:
items['date'] = items['occurred_at'].dt.tz_localize(None).dt.normalize()
items['weekday'] = items['date'].dt.weekday

cutoff = items['date'].max() - pd.Timedelta(days=30)
recent = items[items['date'] >= cutoff]

daily_qty = recent.groupby(['product_id','product_name','date','weekday'])['quantity'].sum().reset_index()
by_weekday = daily_qty.groupby(['product_id','product_name','weekday'])['quantity'].mean().reset_index(name='avg_qty')

tomorrow = (items['date'].max() + pd.Timedelta(days=1))
tomorrow_weekday = tomorrow.weekday()

demand_tomorrow = (by_weekday[by_weekday['weekday'] == tomorrow_weekday]
                    .sort_values('avg_qty', ascending=False)
                    [['product_id','product_name','avg_qty']]
                    .rename(columns={'avg_qty':'expected_qty'}))
demand_tomorrow['expected_qty'] = demand_tomorrow['expected_qty'].round(0)
demand_tomorrow.head(15)

In [ ]:
px.bar(demand_tomorrow.head(20),
       x='expected_qty', y='product_name', orientation='h',
       title=f'Suggested production for {tomorrow:%A %Y-%m-%d}', height=550) \
  .update_yaxes(autorange='reversed')

## 2. Raw material risk — days of cover

For each material, multiply tomorrow's planned production by per-recipe usage to estimate consumption,
then divide on-hand by daily usage to estimate **days of cover**.

In [ ]:
demand = demand_tomorrow.rename(columns={'expected_qty': 'units_per_day'})

consumption = (recipes.merge(demand[['product_id','units_per_day']], on='product_id', how='inner')
                      .assign(daily_use=lambda d: d['quantity'].astype(float) * d['units_per_day'].astype(float))
                      .groupby(['raw_material_id','material_name','material_unit','unit_cost'])
                      ['daily_use'].sum()
                      .reset_index())

stock_rm = stock[stock['kind'] == 'raw_material'][['id','name','quantity','reorder_threshold']]
stock_rm['quantity'] = stock_rm['quantity'].astype(float)

merged = consumption.merge(
    stock_rm.rename(columns={'name':'material_name', 'quantity':'on_hand'}),
    on='material_name', how='left',
).fillna({'on_hand': 0, 'reorder_threshold': 0})

merged['days_of_cover'] = np.where(merged['daily_use'] > 0,
                                    merged['on_hand'] / merged['daily_use'],
                                    np.inf)
merged = merged.sort_values('days_of_cover')
merged[['material_name','material_unit','on_hand','daily_use','days_of_cover','unit_cost']].head(20)

In [ ]:
risk = merged[merged['days_of_cover'] < 10].copy()
risk['order_qty_for_14_days'] = (risk['daily_use'] * 14 - risk['on_hand']).clip(lower=0).round(0)
risk['est_purchase_cost'] = (risk['order_qty_for_14_days'] * risk['unit_cost']).round(0)
risk[['material_name','material_unit','on_hand','days_of_cover','order_qty_for_14_days','est_purchase_cost']]

## 3. Production cost outlook for the next 7 days

In [ ]:
horizon = pd.date_range(items['date'].max() + pd.Timedelta(days=1), periods=7, freq='D')

plan = []
for d in horizon:
    wd = d.weekday()
    expected = (by_weekday[by_weekday['weekday'] == wd]
                 .rename(columns={'avg_qty':'units_per_day'}))[['product_id','units_per_day']]
    cost = (recipes.merge(expected, on='product_id', how='inner')
                   .assign(line_cost=lambda r: r['quantity'].astype(float)
                                                * r['units_per_day'].astype(float)
                                                * r['unit_cost'].astype(float))
                   ['line_cost'].sum())
    plan.append({'day': d, 'est_cogs': round(cost, 2)})

plan_df = pd.DataFrame(plan)
plan_df['rolling'] = plan_df['est_cogs'].cumsum()
plan_df

In [ ]:
px.bar(plan_df, x='day', y='est_cogs', title='Estimated COGS over next 7 days (RWF)')